<a href="https://colab.research.google.com/github/alejandro-cardiles/MLP_PSET_01/blob/main/sam_3_image_segmentation_bbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM3 Image Segmentation for Remote Sensing

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/segment-geospatial/blob/main/docs/examples/sam3_image_segmentation.ipynb)

This notebook demonstrates how to use the Segment Anything Model 3 (SAM3) for segmenting remote sensing images using the `samgeo3` module.

## Installation

First, make sure you have the required dependencies installed:

In [ ]:
%pip install "segment-geospatial[samgeo3]"
%pip install "leafmap"

## Import Libraries

In [ ]:
import leafmap
from samgeo import SamGeo3, download_file
import pandas as pd

## Download Sample Data

Let's download a sample satellite image covering the University of California, Berkeley, for testing:

In [ ]:
url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/uc_berkeley.tif"
image_path = download_file(url)

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite image")
m

In [ ]:
def get_last_drawn_geometry(m):
    features = getattr(m, "draw_features", None)
    if not features:
        raise ValueError("No drawn features found on the map.")
    return features

def extract_geometries_and_bboxes(m):
    features = getattr(m, "draw_features", None)
    if not features:
        return []

    outputs = []

    for i, feat in enumerate(features):
        geom = feat.get("geometry", {})
        gtype = geom.get("type")

        record = {
            "index": i,
            "geometry_type": gtype,
            "geometry": geom,
            "bbox": None
        }

        if gtype == "Point":
            record["bbox"] = geom["coordinates"]

        elif gtype == "Polygon":
            coords = geom["coordinates"][0]   # outer ring
            xs = [p[0] for p in coords]
            ys = [p[1] for p in coords]
            record["bbox"] = [min(xs), min(ys), max(xs), max(ys)]


        outputs.append(record)

    return outputs

In [ ]:
results = extract_geometries_and_bboxes(m)

df = pd.DataFrame(results, columns=["index",
            "geometry_type",
            "geometry",
            "bbox"])
print(df)

# Initialize SAM3


To use SAM3, you need to request access by filling out this form on Hugging Face: https://huggingface.co/facebook/sam3

Once you have access, uncomment the following code block and run it.

When initializing SAM3, you can choose the backend from "meta", or "transformers".

In [ ]:
from huggingface_hub import login
login()

In [ ]:
sam3 = SamGeo3(backend="meta", device=None, checkpoint_path=None, load_from_HF=True,  enable_inst_interactivity=True)

# set image
sam3.set_image(image_path)

# Generate masks with text prompt

In [ ]:
sam3.generate_masks(prompt="trees")
sam3.show_masks()

# Generate masks by bounding boxes

In [ ]:
if (df["geometry_type"] == "Polygon").any():

    print("Polygon")
    boxes = df[df["geometry_type"] == "Polygon"]["bbox"].tolist()
    box_labels = len(boxes) * [True]

else:
    boxes = [[-122.259, 37.8709, -122.2589, 37.871]]

    box_labels = [True]

# Generate masks
sam3.generate_masks_by_boxes(boxes, box_labels, box_crs="EPSG:4326")

In [ ]:
sam3.show_boxes(boxes, box_labels, box_crs="EPSG:4326")

In [ ]:
sam3.show_masks()

In [ ]:
?sam3.generate_masks_by_boxes

# Generate with points

In [ ]:
if (df["geometry_type"] == "Point").any():

    print("Point")
    point = df[df["geometry_type"] == "Point"]["bbox"].tolist()
    point_labels = len(point) * [True]

else:

    point = [[-122.259, 37.8709]]
    point_labels = [True]

# Generate masks
sam3.generate_masks_by_points(point, point_labels, point_crs="EPSG:4326")

In [ ]:
print(point)

In [ ]:
sam3.show_points(point, point_labels, point_crs="EPSG:4326")


In [ ]:
sam3.show_masks()

In [ ]:
?sam3.generate_masks_tiled